In [11]:
pip install "sagemaker==2.*"

Note: you may need to restart the kernel to use updated packages.


In [12]:
import sagemaker
import sagemaker

print(sagemaker)
print(getattr(sagemaker, "__version__", None))
print(list(sagemaker.__path__))

<module 'sagemaker' from '/home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/sagemaker/__init__.py'>
2.257.6
['/home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/sagemaker']


In [13]:
import os
import sagemaker

print("SageMaker path:", list(sagemaker.__path__))

print("\nFiles inside sagemaker:")
print(os.listdir(list(sagemaker.__path__)[0])[:50])

SageMaker path: ['/home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/sagemaker']

Files inside sagemaker:
['serve', 'ai_registry', 'train', 'mlops', '__init__.py', '_studio.py', 'accept_types.py', 'algorithm.py', 'analytics.py', 'base_deserializers.py', 'base_predictor.py', 'base_serializers.py', 'clarify.py', 'collection.py', 'container_base_model.py', 'content_types.py', 'deprecations.py', 'deserializers.py', 'drift_check_baselines.py', 'enums.py', 'environment_variables.py', 'estimator.py', 'exceptions.py', 'fw_utils.py', 'git_utils.py', 'hyperparameters.py', 'image_uris.py', 'inputs.py', 'instance_group.py', 'instance_types.py', 'instance_types_gpu_info.py', 'iterators.py', 'job.py', 'lambda_helper.py', 'logs.py', 'metadata_properties.py', 'metric_definitions.py', 'model.py', 'model_life_cycle.py', 'model_metrics.py', 'model_uris.py', 'multidatamodel.py', 'network.py', 'parameter.py', 'payloads.py', 'pipeline.py', 'predictor.py', 'predictor_async.py', 'proc

In [14]:
from sagemaker.huggingface import HuggingFace

In [15]:
role = sagemaker.get_execution_role()

In [16]:
role

'arn:aws:iam::813667758307:role/SAGEMAKERROLE'

In [17]:
hyperparameters = {
    "model_id": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
    "epochs": 2,
    "per_device_train_batch_size": 2,
    "lr": 2e-5
}

In [18]:
estimator = HuggingFace(
    entry_point="train.py",
    source_dir="./scripts",
    role=role,
    transformers_version="4.36",
    pytorch_version="2.1",
    py_version="py310",
    instance_type="ml.g5.xlarge",
    instance_count=1,
    output_path="s3://llmmodelarti/models/",
    hyperparameters=hyperparameters
) #here iam telling the sagemaker to run the training job in specific instance

## Run only for the training

In [19]:
estimator.fit({
    "train": "s3://llmfine/dataset/"
})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: huggingface-pytorch-training-2026-09-23-08-28-31-921


2026-09-23 08:28:35 Starting - Starting the training job
2026-09-23 08:28:35 Pending - Training job waiting for capacity................................................
2026-09-23 08:36:23 Pending - Preparing the instances for training...
2026-09-23 08:37:06 Downloading - Downloading the training image..................
2026-09-23 08:40:08 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
/opt/conda/lib/python3.10/site-packages/paramiko/pkey.py:100: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
/opt/conda/lib/python3.10/site-packages/paramiko/transport.py:259: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this m

In [20]:
# estimator.latest_training_job.model_data
# estimator.model_data

In [ ]:
# Use this code to check all the accessible services inside your AWS Sagemaker

# from sagemaker import image_uris

# image_uris.retrieve(
#     framework="huggingface",
#     region="ap-south-1",   # change your region
#     version="4.37.0",
#     image_scope="inference"
# )

In [ ]:
 # instance_type="ml.g5.xlarge",

In [ ]:
# model = HuggingFaceModel(
#     model_data="s3://bucket/model.tar.gz",
#     role=role,
#     entry_point="inference.py",
#     source_dir="inference",
#     transformers_version="4.36",
#     pytorch_version="2.1",
#     py_version="py310"
# )

In [3]:
import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")  # Note: your region is us-east-1, not ap-south-1

# Delete endpoint first (must delete endpoint before config)
try:
    sm.delete_endpoint(EndpointName="live-finetune-endpoint")
    print("Endpoint deleted")
except:
    pass

# Delete endpoint config
try:
    sm.delete_endpoint_config(EndpointConfigName="live-finetune-endpoint")
    print("Endpoint config deleted")
except:
    pass

# Now deploy with HF_TASK
import sagemaker
from sagemaker.huggingface import HuggingFaceModel
role = sagemaker.get_execution_role()


model = HuggingFaceModel(
    model_data="s3://llmmodelarti/models/huggingface-pytorch-training-2026-09-23-08-28-31-921/output/model.tar.gz",
    role=role,
    transformers_version="4.37.0",
    pytorch_version="2.1.0",
    py_version="py310",
    env={
        'HF_TASK': 'text-generation'  # Change to your task
    }
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name="live-finetune-endpoint"
)

/home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: HuggingFaceModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
HuggingFaceModel is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


-------!

/home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: HuggingFacePredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
HuggingFacePredictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


In [6]:
## THIS IS JUST TO VALIDATE WHETHER MODEL WORKING OR NOT IN THE NOTEBOOK ITSELF
predictor.predict({"inputs": "Explain AWS S3"})

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 ## THIS IS JUST TO VALIDATE WHETHER MODEL WORKING OR NOT IN THE NOTEBOOK ITSELF              │
│ ❱ 2 predictor.predict({"inputs": "Explain AWS S3"})                                              │
│   3                                                                                              │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/sagemaker/base_predi │
│ ctor.py:222 in predict                                                                           │
│                                                                                                  │
│   219 │   │   if inference_component_name:                                                       │
│   220 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   221 │   │                                                                                      │
│ ❱ 222 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   223 │   │   return self._handle_response(response)                                             │
│   224 │                                                                                          │
│   225 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/botocore/client.py:6 │
│ 06 in _api_call                                                                                  │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/botocore/context.py: │
│ 123 in wrapper                                                                                   │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                            

In [31]:
import boto3
import json

runtime = boto3.client("sagemaker-runtime")

response = runtime.invoke_endpoint(
    EndpointName="live-finetune-endpoint",
    ContentType="application/json",
    Body=json.dumps({
        "inputs": "Explain AWS S3"
    })
)

print(response["Body"].read().decode())

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:6                                                                                    │
│                                                                                                  │
│    3                                                                                             │
│    4 runtime = boto3.client("sagemaker-runtime")                                                 │
│    5                                                                                             │
│ ❱  6 response = runtime.invoke_endpoint(                                                         │
│    7 │   EndpointName="live-finetune-endpoint",                                                  │
│    8 │   ContentType="application/json",                                                         │
│    9 │   Body=json.dumps({                                                                       │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/botocore/client.py:6 │
│ 06 in _api_call                                                                                  │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/botocore/context.py: │
│ 123 in wrapper                                                                                   │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/JupyterSystemEnv/lib/python3.10/site-packages/botocore/client.py:1 │
│ 094 in _make_api_call                                                                            │
│                                                                                                  │
│   1091 │   │   │   │   'error_code_override'                                                     │
│   1092 │   │   │   ) or error_info.get("Code")                                                   │
│   1093 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1094 │   │   │   raise error_class(parsed_response, opera

In [27]:
## after the deployment URL will look like this
https://runtime.sagemaker.<region>.amazonaws.com/endpoints/live-finetune-endpoint/invocations

╭──────────────────────────────────────────────────────────────────────────────────────────────────╮
│ https://runtime.sagemaker.<region>.amazonaws.com/endpoints/live-finetune-endpoint/invocations    │
│       ▲                                                                                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
SyntaxError: invalid syntax

In [4]:
import boto3, json

runtime = boto3.client("sagemaker-runtime", region_name="ap-south-1")

resp = runtime.invoke_endpoint(
    EndpointName="live-finetune-my-endpoint",
    ContentType="application/json",
    Body=json.dumps({"inputs": "hello"})
)

print(resp["Body"].read().decode())


ClientError: An error occurred (UnrecognizedClientException) when calling the InvokeEndpoint operation: The security token included in the request is invalid.